# ML Factory - Google Colab Training

Train all 12 production ML models on financial time-series data using Colab GPU.

## Quick Start
1. **Runtime > Change runtime type > GPU** (T4 or better)
2. **Run Cell 1** (Setup) - clones repo, installs dependencies
3. **Edit Cell 2** (Configuration) - pick models, epochs, features
4. **Run remaining cells** - pipeline runs end-to-end

## Models Available

| Category | Models | GPU Benefit |
|----------|--------|-------------|
| **Boosting** | XGBoost, LightGBM, CatBoost | Minimal (fast on CPU) |
| **Neural RNN** | LSTM, GRU | High |
| **Neural CNN** | TCN, InceptionTime, ResNet1D | High |
| **Transformer** | PatchTST, iTransformer, TFT | Critical (TFT needs GPU) |
| **MLP** | N-BEATS | Moderate |

## Data
Default: MGC (Micro Gold Futures) 1-minute bars with walk-forward validation

In [ ]:
# =============================================================
# CELL 1: SETUP - Run this first
# =============================================================
import os
import sys
import shutil

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    REPO_DIR = "/content/research"

    # Fresh clone
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)

    !git clone https://github.com/Snehpatel101/research.git {REPO_DIR}

    # Install only what Colab doesn't have
    !pip install -q -r {REPO_DIR}/requirements-colab.txt 2>&1 | tail -3

    sys.path.insert(0, REPO_DIR)
    os.chdir(REPO_DIR)

else:
    # Local: assume running from repo root or notebooks/
    REPO_DIR = os.path.dirname(os.path.abspath("."))
    if not os.path.exists(os.path.join(REPO_DIR, "src", "factory.py")):
        REPO_DIR = os.getcwd()
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

# Verify core imports
try:
    from src.factory import MLFactory
    from src.config.experiment import ExperimentConfig
    print("ML Factory loaded successfully!")
except ImportError as e:
    print(f"Import error: {e}")
    raise

# GPU check
import torch

if hasattr(torch, "__version__"):
    print(f"PyTorch: {torch.__version__}")
    if tuple(int(x) for x in torch.__version__.split(".")[:2]) < (2, 0):
        print("WARNING: PyTorch < 2.0 detected. Some models may not work correctly.")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("No GPU detected. Boosting models OK, neural models will be slow.")
    if IN_COLAB:
        print(">> Runtime > Change runtime type > GPU")

print(f"\nRepo: {REPO_DIR}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# =============================================================
# CELL 2: CONFIGURATION - Edit these settings
# =============================================================

# --- DATA ---
SYMBOL = "MGC"
DATA_PATH = f"{REPO_DIR}/data/raw/MGC_1m.parquet"
TARGET_TIMEFRAME = "5min"

# --- MODELS (toggle True/False) ---
# Boosting (fast, ~1-2 min each)
USE_XGBOOST = True
USE_LIGHTGBM = True
USE_CATBOOST = True

# Neural RNN (moderate, ~5-8 min with GPU)
USE_LSTM = True
USE_GRU = True

# Neural CNN (moderate-heavy, 5-30 min with GPU)
USE_TCN = True
USE_INCEPTIONTIME = True
USE_RESNET1D = True

# Transformers (heavy, need GPU)
USE_PATCHTST = True
USE_ITRANSFORMER = True
USE_TFT = True          # Slowest model - ~10h on CPU, ~30min on GPU

# MLP
USE_NBEATS = True

# --- TRAINING ---
HORIZONS = [20]                     # Bars ahead to predict
MAX_EPOCHS = 50                     # 3=quick test, 50=decent, 100=full
EARLY_STOPPING_PATIENCE = 10       # Stop if no improvement for N epochs
BATCH_SIZE = 256
DEVICE = "auto"                     # auto picks GPU if available

# --- WALK-FORWARD VALIDATION ---
# Recommended for >1 year of data. Each window trains on a portion of
# data and tests on the next chunk, sliding forward through time.
# This keeps memory manageable AND gives realistic performance estimates.
TRAINING_MODE = "walk_forward"      # "standard" = single split, "walk_forward" = sliding windows
WF_N_WINDOWS = 5                    # Number of test windows (5 windows on 5yr = ~1yr each)
WF_WINDOW_TYPE = "expanding"        # "expanding" = growing train, "rolling" = fixed-size train
WF_MIN_TRAIN_PCT = 0.4              # Minimum training data (40% of total)
WF_TEST_PCT = 0.1                   # Test data per window (10% of total)

# --- FEATURES ---
MTF_ENABLED = True                  # Multi-timeframe features
MTF_TIMEFRAMES = ["15min", "30min", "1h"]
FEATURE_SELECTION_ENABLED = True
FEATURE_SELECTION_METHOD = "mda"    # mda, mdi, shap, mutual_info

# --- ENSEMBLE ---
BUILD_ENSEMBLE = True
META_LEARNER = "ridge_meta"         # ridge_meta, mlp_meta, xgboost_meta

# --- OPTUNA ---
OPTUNA_ENABLED = True
OPTUNA_TRIALS = 50                  # 0=disable, 25=quick, 50=balanced

# --- EVALUATION ---
RUN_BACKTEST = True
GENERATE_REPORT = True

# --- BUNDLING ---
CREATE_BUNDLE = True                # Create inference bundles for each model
INCLUDE_OOF = False                 # Include OOF predictions in bundle

# --- EXPERIMENT ---
EXPERIMENT_NAME = "mgc_walkforward_12_models"
RANDOM_SEED = 42

# =============================================================
# BUILD MODEL LIST (auto from toggles above)
# =============================================================
MODELS = []
if USE_XGBOOST: MODELS.append("xgboost")
if USE_LIGHTGBM: MODELS.append("lightgbm")
if USE_CATBOOST: MODELS.append("catboost")
if USE_LSTM: MODELS.append("lstm")
if USE_GRU: MODELS.append("gru")
if USE_TCN: MODELS.append("tcn")
if USE_INCEPTIONTIME: MODELS.append("inceptiontime")
if USE_RESNET1D: MODELS.append("resnet1d")
if USE_PATCHTST: MODELS.append("patchtst")
if USE_ITRANSFORMER: MODELS.append("itransformer")
if USE_TFT: MODELS.append("tft")
if USE_NBEATS: MODELS.append("nbeats")

print(f"Models: {len(MODELS)} selected -> {MODELS}")
print(f"Epochs: {MAX_EPOCHS}, Horizons: {HORIZONS}, Device: {DEVICE}")
print(f"Training mode: {TRAINING_MODE}" + (f" ({WF_N_WINDOWS} windows, {WF_WINDOW_TYPE})" if TRAINING_MODE == "walk_forward" else ""))
print(f"Ensemble: {BUILD_ENSEMBLE}, Optuna trials: {OPTUNA_TRIALS if OPTUNA_ENABLED else 'disabled'}")
print(f"Bundling: create={CREATE_BUNDLE}, include_oof={INCLUDE_OOF}")

In [ ]:
# =============================================================
# CELL 3: VALIDATE CONFIGURATION
# =============================================================
import os
import torch

VALID_MODELS = {
    "xgboost", "lightgbm", "catboost",
    "lstm", "gru", "tcn", "nbeats",
    "inceptiontime", "resnet1d",
    "patchtst", "itransformer", "tft",
}
NEURAL_MODELS = {
    "lstm", "gru", "tcn", "nbeats",
    "inceptiontime", "resnet1d",
    "patchtst", "itransformer", "tft",
}

errors, warnings = [], []

# Model check
for m in MODELS:
    if m not in VALID_MODELS:
        errors.append(f"Unknown model: '{m}'")
if not MODELS:
    errors.append("No models selected!")

# Data check
if not os.path.exists(DATA_PATH):
    errors.append(f"Data file not found: {DATA_PATH}")

# Training mode check
if TRAINING_MODE not in ("standard", "walk_forward", "regime_aware", "meta_labeling"):
    errors.append(f"Unknown training mode: '{TRAINING_MODE}'")

if TRAINING_MODE == "walk_forward":
    if WF_MIN_TRAIN_PCT + WF_N_WINDOWS * WF_TEST_PCT > 1.0:
        errors.append(
            f"Walk-forward config invalid: min_train_pct ({WF_MIN_TRAIN_PCT}) + "
            f"n_windows ({WF_N_WINDOWS}) * test_pct ({WF_TEST_PCT}) > 1.0"
        )
    print(f"Walk-forward: {WF_N_WINDOWS} windows, {WF_WINDOW_TYPE}, "
          f"min_train={WF_MIN_TRAIN_PCT*100:.0f}%, test={WF_TEST_PCT*100:.0f}% per window")

# GPU check for neural models
selected_neural = [m for m in MODELS if m in NEURAL_MODELS]
if selected_neural and not torch.cuda.is_available():
    warnings.append(
        f"No GPU but neural models selected: {selected_neural}. "
        "Will be slow. Runtime > Change runtime type > GPU"
    )

if "tft" in MODELS and not torch.cuda.is_available():
    warnings.append("TFT without GPU will take ~10 hours. Consider disabling it.")

# Memory warning for walk-forward + many models
if TRAINING_MODE == "walk_forward" and len(MODELS) >= 8:
    est_gb = len(MODELS) * WF_N_WINDOWS * 0.5  # ~0.5 GB per model per window
    warnings.append(
        f"Walk-forward with {len(MODELS)} models x {WF_N_WINDOWS} windows "
        f"may need ~{est_gb:.0f} GB RAM. "
        "If OOM: reduce models, windows, or use 'standard' mode first."
    )

# Report
if errors:
    for e in errors:
        print(f"ERROR: {e}")
    raise ValueError("Fix errors above in Cell 2")

if warnings:
    for w in warnings:
        print(f"WARNING: {w}")

print(f"\nConfig OK: {len(MODELS)} models, data exists, device={DEVICE}, mode={TRAINING_MODE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [4]:
# =============================================================
# CELL 4: LOAD & PREVIEW DATA
# =============================================================
import pandas as pd

# Load data based on file extension
if DATA_PATH.endswith(".parquet"):
    raw_data = pd.read_parquet(DATA_PATH)
elif DATA_PATH.endswith(".csv"):
    raw_data = pd.read_csv(DATA_PATH)
else:
    raise ValueError(f"Unsupported file format: {DATA_PATH}. Use .parquet or .csv")

# Normalize column names to lowercase
raw_data.columns = [c.lower().strip() for c in raw_data.columns]

# Validate required OHLCV columns
REQUIRED_COLUMNS = ["open", "high", "low", "close", "volume"]
missing = [c for c in REQUIRED_COLUMNS if c not in raw_data.columns]
if missing:
    raise ValueError(
        f"Missing required OHLCV columns: {missing}\n"
        f"Found columns: {list(raw_data.columns)}\n"
        f"The pipeline expects: {REQUIRED_COLUMNS}"
    )

# Ensure datetime index
if "datetime" in raw_data.columns:
    raw_data["datetime"] = pd.to_datetime(raw_data["datetime"])
    raw_data = raw_data.set_index("datetime").sort_index()
elif "date" in raw_data.columns:
    raw_data["date"] = pd.to_datetime(raw_data["date"])
    raw_data = raw_data.set_index("date").sort_index()
    raw_data.index.name = "datetime"
elif not isinstance(raw_data.index, pd.DatetimeIndex):
    # Try parsing the existing index as datetime
    try:
        raw_data.index = pd.to_datetime(raw_data.index)
        raw_data.index.name = "datetime"
        raw_data = raw_data.sort_index()
    except Exception:
        raise ValueError(
            "Could not find or parse a datetime column. "
            "Data must have a 'datetime' or 'date' column, or a datetime-parseable index."
        )

# --- Summary ---
print("=" * 50)
print("Data Loaded Successfully")
print("=" * 50)
print(f"  Symbol:      {SYMBOL}")
print(f"  Rows:        {len(raw_data):,}")
print(f"  Shape:       {raw_data.shape}")
print(f"  Columns:     {list(raw_data.columns)}")
print(f"  Date range:  {raw_data.index.min()} -> {raw_data.index.max()}")
print(f"  Index name:  {raw_data.index.name}")
print()

# Missing values
missing_counts = raw_data[REQUIRED_COLUMNS].isnull().sum()
total_missing = missing_counts.sum()
if total_missing > 0:
    print("WARNING: Missing values in OHLCV columns:")
    for col, count in missing_counts.items():
        if count > 0:
            print(f"  {col}: {count} ({count/len(raw_data)*100:.2f}%)")
else:
    print("No missing values in OHLCV columns.")
print()

# Preview
print("First 5 rows:")
display(raw_data.head())

print(f"\nData ready: 'raw_data' DataFrame with {len(raw_data):,} rows.")

In [ ]:
# =============================================================
# CELL 5: RUN ML FACTORY
# =============================================================
from src.config.experiment import (
    ExperimentConfig,
    DataSection,
    TrainingSection,
    EvaluationSection,
)
from src.config.training import OptunaConfig
from src.config.data import FeatureConfig, LabelingConfig, MTFConfig
from src.factory import MLFactory

config = ExperimentConfig(
    name=EXPERIMENT_NAME,
    random_seed=RANDOM_SEED,
    verbose=1,

    data=DataSection(
        symbol=SYMBOL,
        data_path=DATA_PATH,
        features=FeatureConfig(
            mode="full",
            selection_enabled=FEATURE_SELECTION_ENABLED,
            selection_method=FEATURE_SELECTION_METHOD if FEATURE_SELECTION_ENABLED else "mda",
        ),
        labeling=LabelingConfig(method="triple_barrier"),
        mtf=MTFConfig(
            enabled=MTF_ENABLED,
            mode="indicators" if MTF_ENABLED else "none",
            timeframes=MTF_TIMEFRAMES if MTF_ENABLED else [],
            primary_timeframe=TARGET_TIMEFRAME,
        ),
    ),

    training=TrainingSection(
        models=MODELS,
        horizons=HORIZONS,
        training_mode=TRAINING_MODE,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        build_ensemble=BUILD_ENSEMBLE,
        meta_learner=META_LEARNER if BUILD_ENSEMBLE else "ridge_meta",
        optuna=OptunaConfig(
            n_trials=OPTUNA_TRIALS if OPTUNA_ENABLED else 1,
        ),
    ),

    evaluation=EvaluationSection(
        run_backtest=RUN_BACKTEST,
        generate_report=GENERATE_REPORT,
    ),
)

print(f"Experiment: {config.name}")
print(f"Models: {config.training.models}")
print(f"Training mode: {config.training.training_mode}")
print(f"Epochs: {config.training.max_epochs}, Device: {config.training.device}")
print()

factory = MLFactory(config, enable_checkpoints=True)

try:
    result = factory.run()
    print()
    if result.success:
        print(result.summary())
    else:
        print(f"Pipeline completed with errors: {result.error_message}")
except KeyboardInterrupt:
    print("\nInterrupted. Resume with: result = factory.resume_from_checkpoint()")
    result = None
except Exception as e:
    print(f"\nERROR: {e}")
    import traceback
    traceback.print_exc()
    print("\nResume with: result = factory.resume_from_checkpoint()")
    result = None

In [ ]:
# =============================================================
# CELL 6: RESULTS & VISUALIZATION
# =============================================================
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import glob

if "result" not in dir() or result is None or not result.success:
    msg = "No successful result to display."
    if "result" in dir() and result and result.error_message:
        msg += f"\nError: {result.error_message}"
    print(msg)
else:
    print("=" * 60)
    print("EXPERIMENT RESULTS")
    print("=" * 60)
    print(f"  Run ID:          {result.run_id}")
    print(f"  Models trained:  {result.n_models}")
    print(f"  Best model:      {result.best_model}")
    print(f"  Duration:        {result.duration_seconds:.1f}s ({result.duration_seconds/60:.1f} min)")
    print()

    # --- Model Metrics Table ---
    if result.metrics:
        print("-" * 40)
        print("Model Performance")
        print("-" * 40)
        metrics_df = pd.DataFrame(result.metrics).T
        metrics_df.index.name = "model"
        display(metrics_df.round(4))
        print()

    # --- Ensemble Metrics ---
    if result.ensemble_metrics:
        print("-" * 40)
        print("Ensemble Metrics")
        print("-" * 40)
        for k, v in result.ensemble_metrics.items():
            if isinstance(v, float):
                print(f"  {k}: {v:.4f}")
            else:
                print(f"  {k}: {v}")
        print()

    # --- Backtest Metrics ---
    if result.backtest_metrics:
        print("-" * 40)
        print("Backtest Results")
        print("-" * 40)
        highlight_keys = ["sharpe_ratio", "max_drawdown_pct", "profit_factor", "win_rate_pct"]
        for k in highlight_keys:
            if k in result.backtest_metrics:
                print(f"  {k}: {result.backtest_metrics[k]}")
        for k, v in result.backtest_metrics.items():
            if k not in highlight_keys:
                if isinstance(v, (int, float)):
                    print(f"  {k}: {v}")
        print()

    # --- Display Plot Images ---
    if result.output_dir and Path(result.output_dir).exists():
        plot_files = sorted(glob.glob(str(Path(result.output_dir) / "**" / "*.png"), recursive=True))[:6]
        if plot_files:
            print("-" * 40)
            print(f"Plots ({len(plot_files)} found)")
            print("-" * 40)
            n_plots = len(plot_files)
            cols = min(n_plots, 2)
            rows = (n_plots + cols - 1) // cols
            fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 5 * rows))
            if n_plots == 1:
                axes = [axes]
            else:
                axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
            for i, pf in enumerate(plot_files):
                img = mpimg.imread(pf)
                axes[i].imshow(img)
                axes[i].set_title(Path(pf).stem, fontsize=10)
                axes[i].axis("off")
            # Hide unused subplots
            for j in range(n_plots, len(axes)):
                axes[j].axis("off")
            plt.tight_layout()
            plt.show()

    if result.bundle_path:
        print(f"Bundle path: {result.bundle_path}")
    if result.output_dir:
        print(f"Output dir:  {result.output_dir}")

In [ ]:
# =============================================================
# CELL 7: DEPLOY ARTIFACT - Production inference entry point
# =============================================================
#
# After training, the factory creates a deploy/ directory with a
# JSON manifest indexing all bundles by horizon. This cell shows
# how to load and use the deploy artifact for inference.
#
# Usage:
#   artifact = load_deploy_artifact("./deploy", horizon=20)
#   pred = artifact.predict_from_raw(raw_bars_df)

from pathlib import Path
import pandas as pd

if "result" not in dir() or result is None or not result.success:
    print("No successful result. Run Cell 5 first.")
elif result.deploy_path and Path(result.deploy_path).exists():
    from src.inference.deploy import (
        load_deploy_artifact,
        validate_deploy_artifact,
        DeployManifest,
        DEPLOY_MANIFEST_FILE,
    )

    deploy_dir = Path(result.deploy_path)

    # --- Validate the deploy artifact ---
    validation = validate_deploy_artifact(deploy_dir)
    print("=" * 50)
    print("DEPLOY ARTIFACT")
    print("=" * 50)
    print(f"  Path:     {deploy_dir}")
    print(f"  Valid:    {validation['valid']}")
    print(f"  Horizons: {validation.get('n_horizons', 0)}")
    if validation["issues"]:
        for issue in validation["issues"]:
            print(f"  ISSUE: {issue}")
    print()

    # --- Show manifest contents ---
    manifest = DeployManifest.load(deploy_dir / DEPLOY_MANIFEST_FILE)
    for h, h_manifest in sorted(manifest.horizons.items()):
        print(f"  Horizon {h}:")
        print(f"    Primary: {h_manifest.primary_model}")
        for entry in h_manifest.entries:
            tag = " [ensemble]" if entry.is_ensemble else ""
            print(f"    - {entry.model_name}{tag} -> {entry.bundle_path}")
    print()

    # --- Load the primary artifact for inference ---
    horizon = HORIZONS[0] if HORIZONS else 20
    try:
        artifact = load_deploy_artifact(deploy_dir, horizon=horizon)
        print(f"Loaded artifact for H{horizon}: {type(artifact).__name__}")
        print(f"  Model:  {getattr(artifact.metadata, 'model_name', getattr(artifact.metadata, 'meta_learner_name', 'N/A'))}")
        print()
        print("Ready for inference:")
        print("  pred = artifact.predict_from_raw(raw_bars_df)")
    except Exception as e:
        print(f"Could not load artifact: {e}")
        print("(This is normal if bundles were not saved to deploy dir)")

    # --- Run predict_from_raw() on a sample of raw data ---
    print()
    print("-" * 50)
    print("INFERENCE DEMO: predict_from_raw()")
    print("-" * 50)
    try:
        # Use last 500 bars as a sample (enough for indicator warm-up)
        sample = raw_data.tail(500).copy()
        print(f"  Sample: {len(sample)} bars ({sample.index.min()} -> {sample.index.max()})")

        pred = artifact.predict_from_raw(sample)

        # Extract class predictions
        if hasattr(pred, "class_predictions"):
            classes = pd.Series(pred.class_predictions)
        elif hasattr(pred, "predictions") and hasattr(pred.predictions, "class_predictions"):
            classes = pd.Series(pred.predictions.class_predictions)
        elif isinstance(pred, pd.DataFrame) and "prediction" in pred.columns:
            classes = pred["prediction"]
        else:
            classes = pd.Series(pred) if not isinstance(pred, pd.Series) else pred

        # Class distribution
        label_map = {-1: "Short", 0: "Neutral", 1: "Long"}
        dist = classes.value_counts().sort_index()
        print(f"\n  Predictions: {len(classes)} samples")
        print("  Class distribution:")
        for cls_val, count in dist.items():
            label = label_map.get(int(cls_val), str(cls_val))
            pct = count / len(classes) * 100
            print(f"    {label:>8s} ({int(cls_val):+d}): {count:>5d}  ({pct:5.1f}%)")

        # Confidence stats if available
        if hasattr(pred, "confidence"):
            conf = pred.confidence
        elif hasattr(pred, "predictions") and hasattr(pred.predictions, "confidence"):
            conf = pred.predictions.confidence
        else:
            conf = None
        if conf is not None:
            import numpy as np
            conf = np.asarray(conf)
            print(f"\n  Confidence: mean={conf.mean():.3f}, std={conf.std():.3f}, "
                  f"min={conf.min():.3f}, max={conf.max():.3f}")

        print("\n  Inference demo complete.")

    except Exception as e:
        print(f"  predict_from_raw() failed: {e}")
        print("  (Expected if preprocessing graph was not saved with bundle)")
else:
    print("No deploy artifact found.")
    print("Ensure BundlingSection.deploy_artifact=True in config (default).")

In [ ]:
# =============================================================
# CELL 7b: GOOGLE DRIVE MOUNT HELPER
# =============================================================
# Mount Google Drive to persist bundles and results across sessions.
# Skipped automatically when running locally.

from pathlib import Path
import shutil

DRIVE_DEST = "ml_factory_results"  # Folder name in your Drive root

if IN_COLAB:
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        drive_root = Path("/content/drive/MyDrive") / DRIVE_DEST
        drive_root.mkdir(parents=True, exist_ok=True)

        print(f"Google Drive mounted. Save path: {drive_root}")

        # Copy bundles to Drive if they exist
        if "result" in dir() and result is not None and result.success:
            # Copy deploy artifact
            if result.deploy_path and Path(result.deploy_path).exists():
                dest = drive_root / EXPERIMENT_NAME / "deploy"
                if dest.exists():
                    shutil.rmtree(dest)
                shutil.copytree(result.deploy_path, dest)
                print(f"  Deploy artifact saved to Drive: {dest}")

            # Copy output dir
            if result.output_dir and Path(result.output_dir).exists():
                dest = drive_root / EXPERIMENT_NAME / "output"
                if dest.exists():
                    shutil.rmtree(dest)
                shutil.copytree(result.output_dir, dest)
                print(f"  Output dir saved to Drive: {dest}")

            print(f"\nResults persisted to Google Drive: {drive_root / EXPERIMENT_NAME}")
        else:
            print("No results to save yet. Run Cell 5 first, then re-run this cell.")

    except ImportError:
        print("google.colab not available. Running locally?")
    except Exception as e:
        print(f"Drive mount failed: {e}")
        print("You can manually mount with: from google.colab import drive; drive.mount('/content/drive')")
else:
    print("Not in Colab - skipping Drive mount. Results saved locally.")

In [ ]:
# =============================================================
# CELL 9: INFERENCE-ONLY EXPORT
# =============================================================
# Zip only the deploy/ or bundles/ directory for lightweight
# deployment. Excludes training logs, plots, and checkpoints.

from pathlib import Path
import shutil

if "result" not in dir() or result is None or not result.success:
    print("No successful result. Run Cell 5 first.")
else:
    output_dir = Path(result.output_dir) if result.output_dir else None
    deploy_dir = Path(result.deploy_path) if result.deploy_path else None

    # Prefer deploy/ artifact (self-contained), fall back to bundles/
    if deploy_dir and deploy_dir.exists():
        export_src = deploy_dir
        export_label = "deploy"
    elif output_dir:
        # Look for bundles directory inside output
        bundles_dir = output_dir / "bundles"
        if bundles_dir.exists():
            export_src = bundles_dir
            export_label = "bundles"
        else:
            export_src = None
            export_label = None
    else:
        export_src = None
        export_label = None

    if export_src is None:
        print("No deploy/ or bundles/ directory found to export.")
    else:
        zip_name = f"{EXPERIMENT_NAME}_inference_only"
        if IN_COLAB:
            zip_path = f"/content/{zip_name}"
        else:
            zip_path = str(output_dir.parent / zip_name) if output_dir else zip_name

        shutil.make_archive(zip_path, "zip", export_src)
        zip_file = Path(f"{zip_path}.zip")

        print("=" * 50)
        print("INFERENCE-ONLY EXPORT")
        print("=" * 50)
        print(f"  Source:   {export_src} ({export_label})")
        print(f"  Archive:  {zip_file}")
        print(f"  Size:     {zip_file.stat().st_size / 1e6:.1f} MB")

        # Compare with full output size
        if output_dir and output_dir.exists():
            full_size = sum(f.stat().st_size for f in output_dir.rglob("*") if f.is_file())
            infer_size = zip_file.stat().st_size
            print(f"\n  Full output:     {full_size / 1e6:.1f} MB")
            print(f"  Inference-only:  {infer_size / 1e6:.1f} MB")
            if full_size > 0:
                print(f"  Reduction:       {(1 - infer_size / full_size) * 100:.0f}%")

        # Auto-download in Colab
        if IN_COLAB:
            try:
                from google.colab import files
                files.download(str(zip_file))
            except Exception:
                print(f"\nManual download: files.download('{zip_file}')")

        print(f"\nTo use: unzip {zip_file.name} and load with load_deploy_artifact()")

In [ ]:
# =============================================================
# CELL 7b: GOOGLE DRIVE MOUNT - Save bundles to Drive
# =============================================================
# Mount Google Drive in Colab so bundles persist across sessions.
# Skip this cell if running locally.

from pathlib import Path

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_BUNDLES = Path("/content/drive/MyDrive/ml_factory_bundles")
        DRIVE_BUNDLES.mkdir(parents=True, exist_ok=True)
        print(f"Drive mounted. Bundles will be saved to: {DRIVE_BUNDLES}")

        # Copy bundles to Drive if they exist
        if "result" in dir() and result and result.bundle_path:
            import shutil
            src_bundles = Path(result.bundle_path)
            if src_bundles.exists():
                dest = DRIVE_BUNDLES / EXPERIMENT_NAME
                if dest.exists():
                    shutil.rmtree(dest)
                shutil.copytree(src_bundles, dest)
                print(f"Copied bundles to Drive: {dest}")
                n_files = sum(1 for f in dest.rglob("*") if f.is_file())
                size_mb = sum(f.stat().st_size for f in dest.rglob("*") if f.is_file()) / 1e6
                print(f"  Files: {n_files}, Size: {size_mb:.1f} MB")
        else:
            print("No bundles to copy yet. Run Cell 5 first.")

    except ImportError:
        print("Not in Colab, skipping Drive mount.")
else:
    print("Local environment. Bundles are saved in the output directory.")

In [ ]:
# =============================================================
# CELL 8: SAVE & DOWNLOAD RESULTS
# =============================================================
from pathlib import Path
import shutil

if "result" not in dir() or result is None or not result.success:
    print("No successful result to save.")
elif result.output_dir and Path(result.output_dir).exists():
    src_dir = Path(result.output_dir)

    if IN_COLAB:
        # Zip results for download
        zip_path = f"/content/{EXPERIMENT_NAME}_results"
        shutil.make_archive(zip_path, "zip", src_dir)
        print(f"Results zipped: {zip_path}.zip")
        print(f"Size: {Path(zip_path + '.zip').stat().st_size / 1e6:.1f} MB")

        # Auto-download in Colab
        try:
            from google.colab import files
            files.download(f"{zip_path}.zip")
        except Exception:
            print(f"\nManual download: files.download('{zip_path}.zip')")

        # Also try saving to Drive if mounted
        drive_path = Path("/content/drive/MyDrive/ml_factory_results")
        if drive_path.parent.exists():
            save_dest = drive_path / EXPERIMENT_NAME
            if save_dest.exists():
                shutil.rmtree(save_dest)
            shutil.copytree(src_dir, save_dest)
            print(f"\nAlso saved to Drive: {save_dest}")
    else:
        print(f"Results at: {src_dir}")
        n_files = sum(1 for f in src_dir.rglob("*") if f.is_file())
        print(f"Files: {n_files}")
else:
    print("No output directory found.")

In [ ]:
# =============================================================
# CELL 9: INFERENCE-ONLY EXPORT
# =============================================================
# Export just the bundles/ directory (no training artifacts, OOF, etc.)
# This is the minimal package needed for production inference.

from pathlib import Path
import shutil

if "result" not in dir() or result is None or not result.success:
    print("No successful result. Run Cell 5 first.")
elif result.output_dir and Path(result.output_dir).exists():
    output_dir = Path(result.output_dir)

    # Find bundle directories
    bundle_dirs = []
    if result.bundle_path and Path(result.bundle_path).exists():
        bundle_dirs.append(Path(result.bundle_path))
    if result.deploy_path and Path(result.deploy_path).exists():
        bundle_dirs.append(Path(result.deploy_path))

    if not bundle_dirs:
        # Fallback: look for bundles/ subdirectory
        for candidate in ["bundles", "deploy"]:
            p = output_dir / candidate
            if p.exists():
                bundle_dirs.append(p)

    if not bundle_dirs:
        print("No bundle directories found in output.")
    else:
        # Create inference-only zip
        inference_dir = Path(f"/tmp/{EXPERIMENT_NAME}_inference")
        if inference_dir.exists():
            shutil.rmtree(inference_dir)
        inference_dir.mkdir(parents=True)

        for bd in bundle_dirs:
            dest = inference_dir / bd.name
            shutil.copytree(bd, dest)

        zip_path = f"/tmp/{EXPERIMENT_NAME}_inference"
        shutil.make_archive(zip_path, "zip", inference_dir)

        full_size = sum(f.stat().st_size for f in output_dir.rglob("*") if f.is_file())
        inf_size = Path(f"{zip_path}.zip").stat().st_size

        print("=" * 50)
        print("INFERENCE-ONLY EXPORT")
        print("=" * 50)
        print(f"  Full output:      {full_size / 1e6:.1f} MB")
        print(f"  Inference-only:   {inf_size / 1e6:.1f} MB")
        if full_size > 0:
            print(f"  Size reduction:   {(1 - inf_size / full_size) * 100:.0f}%")
        print(f"  Contents:         {[bd.name for bd in bundle_dirs]}")
        print(f"  Zip path:         {zip_path}.zip")

        if IN_COLAB:
            try:
                from google.colab import files
                files.download(f"{zip_path}.zip")
            except Exception:
                print(f"\nManual download: files.download('{zip_path}.zip')")

        # Cleanup temp
        shutil.rmtree(inference_dir)
else:
    print("No output directory found.")